In [1]:
!pip install tensorflow datasets gensim sacrebleu rouge-score scikit-learn
!git clone https://github.com/festvox/datasets-CMU_DoG

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=3eb9068f5e85a6f0eecef669b1bd1d93926d4e10c251f1825c3173cf974bc8f7
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score
Cloning into 'datasets-CMU_DoG'...
remote: Enumerating objects: 8351, done.
remote: Total 8351 (delta 0), reused 0 (delta 0), pack-reused 8351 (from 1)
Receiving objects: 100% (8351/8351), 10.88 MiB | 13.58 MiB/s, done.
Resolving deltas: 100% (7491/7491), done.


In [2]:
## import some useful libraries
import os, re, json, random, collections
import numpy as np
import tensorflow as tf
from datasets import load_dataset

In [3]:
## variable fixed before stsrting
CFG = dict(
    doc_dir="datasets-CMU_DoG/WikiData",   # <-- CHANGE: folder with 0.json..29.json
    hist_turns=3,        # how many previous turns form the dialogue history
    max_hist_len=80,     # tokens (keep the most recent)
    max_doc_len=200,     # tokens (current wiki section, truncated)
    max_resp_len=30,     # tokens in the reply
    vocab_size=12000,
    emb_dim=128,
    units=256,
    dropout=0.3,
    batch=64,
    epochs=20,
    lr=1e-3,
    seed=42,
)
PAD, UNK, BOS, EOS, SEP, S1, S2 = range(7)
SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>", "<sep>", "<s1>", "<s2>"]
SPK = {"user1": "<s1>", "user2": "<s2>"}

#### Load data

In [4]:
## make a function for load file instead of load file every time manually
def load_doc(doc_dir):
  docs = {}

  for fn in os.listdir(doc_dir):
      if not fn.endswith(".json"):
          continue

      path = os.path.join(doc_dir, fn)

      with open(path, encoding="utf-8") as f:
          data = json.load(f)

      if "wikiDocumentIdx" not in data:
          print("Skipping:", fn, "- no wikiDocumentIdx")
          continue

      doc_id = data["wikiDocumentIdx"]
      docs[doc_id] = data

  return docs

In [5]:
## now starts loading wikidocs data
def flatten(x):
    if isinstance(x, str):
        return x
    if isinstance(x, dict):
        return " ".join(flatten(v) for v in x.values())
    if isinstance(x, (list, tuple)):
        return " ".join(flatten(v) for v in x)
    return str(x)

In [6]:
def section_text(doc, sec):
    """Safely extract the required wiki section."""

    if isinstance(doc, dict):
        # Try string key
        if str(sec) in doc:
            return flatten(doc[str(sec)])

        # Try integer key
        if sec in doc:
            return flatten(doc[sec])

        # If section doesn't exist, use whole document
        return flatten(doc)

    # If document is a list
    if isinstance(doc, list):
        if 0 <= int(sec) < len(doc):
            return flatten(doc[int(sec)])
        return flatten(doc)

    # If document is already a string
    return flatten(doc)

In [7]:
def inspect_docs(doc_dir=None):
    docs = load_documents(doc_dir or CFG["doc_dir"])
    k = sorted(docs)[0]
    print("n docs:", len(docs), "| first doc type:", type(docs[k]))
    print(json.dumps(docs[k], indent=1)[:1500])

In [8]:
def load_conversations(split):
    """HF rows are single utterances. Rebuild conversations by grouping."""
    ds = load_dataset("festvox/cmu_hinglish_dog", split=split)
    convs = collections.defaultdict(list)
    for r in ds:
        key = (r["date"], r["user2_id"], r["wikiDocumentIdx"])
        convs[key].append(dict(
            ts=r["utcTimestamp"], uid=r["uid"], sec=r["docIdx"], doc=r["wikiDocumentIdx"],
            en=r["translation"]["en"], hi_en=r["translation"]["hi_en"],
            who_saw=r["whoSawDoc"]))
    for k in convs:
        convs[k].sort(key=lambda t: t["ts"])      # restore turn order
    return convs

#### preprocessing and cleaning data


In [9]:
TOKEN_RE = re.compile(r"\w+(?:'\w+)?|[^\s\w]", re.UNICODE)

In [10]:
## cleaning
def clean(s):
    s = (s or "").lower().strip()  ## convert lower case
    s = re.sub(r"https?://\S+", " url ", s) ## remove given patterns
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)          # sooooo -> soo
    return s

In [11]:
## tokenize after cleaning

def tok(s):
    return TOKEN_RE.findall(clean(s))

In [12]:
## encode and decode words from the laibrary

class Vocab:
    def __init__(self, utt_tokens, doc_tokens, max_size, min_freq=2):
        cu, cd = collections.Counter(), collections.Counter()
        for t in utt_tokens: cu.update(t)
        for t in doc_tokens: cd.update(t)
        total = cu + cd
        # keep every doc word (names/titles matter) + utterance words seen >= min_freq
        words = [w for w, _ in total.most_common() if cu[w] >= min_freq or cd[w] >= 1]
        self.itos = SPECIALS + [w for w in words if w not in SPECIALS][: max_size - len(SPECIALS)]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.common = {w for w, _ in cu.most_common(150)}   # "stopword-ish" for grounding metric

    def enc(self, toks):
        return [self.stoi.get(t, UNK) for t in toks]

    def dec(self, ids):
        out = []
        for i in ids:
            if i == EOS: break
            if i > S2 or i == UNK: out.append(self.itos[i])
        return " ".join(out)

In [44]:
## tells the history of previous conversation
def build_examples(convs, docs, hist_turns):
    cache, ex = {}, []
    for turns in convs.values():
        for t in range(1, len(turns)):
            cur = turns[t]
            resp = tok(cur["hi_en"])
            if not resp:
                continue
            hist = []
            for h in turns[max(0, t - hist_turns): t]:
                hist += [SPK[h["uid"]]] + tok(h["en"]) + ["<sep>"]
            hist.append(SPK[cur["uid"]])               # tell the model who speaks next
            k = (cur["doc"], cur["sec"])
            if k not in cache:
                cache[k] = tok(section_text(docs[cur["doc"]], cur["sec"]))
            ex.append(dict(hist=hist, doc=cache[k], resp=resp, en=cur["en"],
                           grounded=cur["uid"] in cur["who_saw"]))   # did the speaker see the doc?
    return ex

In [45]:
def pad(seqs, L, keep_end=False):
    a = np.zeros((len(seqs), L), dtype="int32")
    for i, s in enumerate(seqs):
        s = s[-L:] if keep_end else s[:L]
        a[i, :len(s)] = s
    return a

In [46]:
## encode the conversation
def encode_examples(ex, vocab, cfg):
    Lr = cfg["max_resp_len"]
    resp = [vocab.enc(e["resp"])[:Lr] for e in ex]
    return dict(
        hist=pad([vocab.enc(e["hist"]) for e in ex], cfg["max_hist_len"], keep_end=True),
        doc=pad([vocab.enc(e["doc"]) for e in ex], cfg["max_doc_len"]),
        dec_in=pad([[BOS] + r for r in resp], Lr + 1),
        dec_out=pad([r + [EOS] for r in resp], Lr + 1),
    )

In [47]:
def make_ds(a, batch, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(((a["hist"], a["doc"], a["dec_in"]), a["dec_out"]))
    if shuffle:
        ds = ds.shuffle(10000, seed=CFG["seed"])
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

#### Embedding

In [48]:
from gensim.models import FastText

In [49]:
def build_embedding_matrix(sentences, vocab, dim):
    rng = np.random.default_rng(CFG["seed"])
    mat = rng.normal(0, 0.05, (len(vocab.itos), dim)).astype("float32")

    ft = FastText(sentences=sentences, vector_size=dim, window=5, min_count=1,
                  sg=1, min_n=2, max_n=5, epochs=20, seed=CFG["seed"], workers=1)
    for i, w in enumerate(vocab.itos):
        if i > S2:
            mat[i] = ft.wv[w]
    mat[PAD] = 0.0
    return mat

#### MODEL: two encoders + decoder attending over both sources

In [50]:
L = tf.keras.layers

In [51]:
class BahdanauAttention(L.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1, self.W2, self.V = L.Dense(units), L.Dense(units), L.Dense(1)

    def call(self, query, values, mask):
        score = tf.squeeze(self.V(tf.nn.tanh(self.W1(values) + self.W2(query[:, None, :]))), -1)
        score += (1.0 - tf.cast(mask, score.dtype)) * -1e9        # ignore padding
        w = tf.nn.softmax(score, axis=-1)                          # [B, T]
        ctx = tf.reduce_sum(w[:, :, None] * values, axis=1)        # [B, units]
        return ctx, w

In [52]:
class Encoder(L.Layer):
    """BiGRU over a token sequence. Embedding layer is shared with the decoder."""
    def __init__(self, emb, units, dropout):
        super().__init__()
        self.emb = emb
        self.drop = L.Dropout(dropout)
        self.rnn = L.Bidirectional(L.GRU(units // 2, return_sequences=True, return_state=True))

    def call(self, x, training=False):
        mask = tf.not_equal(x, PAD)
        out, fh, bh = self.rnn(self.drop(self.emb(x), training=training), mask=mask, training=training)
        return out, tf.concat([fh, bh], -1), mask

In [53]:
## final model class


class DoGModel(tf.keras.Model):
    def __init__(self, vocab_size, emb_dim, units, dropout, emb_matrix=None, use_doc=True):
        super().__init__()
        self.use_doc = use_doc
        init = tf.keras.initializers.Constant(emb_matrix) if emb_matrix is not None else "uniform"
        self.emb = L.Embedding(vocab_size, emb_dim, embeddings_initializer=init)  # shared (fine-tuned)
        self.hist_enc = Encoder(self.emb, units, dropout)
        if use_doc:
            self.doc_enc = Encoder(self.emb, units, dropout)
            self.attn_d = BahdanauAttention(units)
        self.attn_h = BahdanauAttention(units)
        self.init_proj = L.Dense(units, activation="tanh")
        self.cell = L.GRUCell(units)
        self.drop = L.Dropout(dropout)
        self.mix = L.Dense(units, activation="tanh")
        self.out = L.Dense(vocab_size)

    def encode(self, hist, doc, training):
        h_out, h_state, h_mask = self.hist_enc(hist, training=training)
        if self.use_doc:
            d_out, d_state, d_mask = self.doc_enc(doc, training=training)
            state = self.init_proj(tf.concat([h_state, d_state], -1))
        else:
            d_out = d_mask = None
            state = self.init_proj(h_state)
        return (h_out, h_mask, d_out, d_mask), state

    def step(self, y_prev, state, enc, training):
        h_out, h_mask, d_out, d_mask = enc
        ctx_h, _ = self.attn_h(state, h_out, h_mask)
        ctxs = [ctx_h]
        w_d = None
        if self.use_doc:
            ctx_d, w_d = self.attn_d(state, d_out, d_mask)         # second attention: the document
            ctxs.append(ctx_d)
        x = tf.concat([self.drop(self.emb(y_prev), training=training)] + ctxs, -1)
        h, new_states = self.cell(x, [state])
        state = new_states[0]
        feat = self.mix(tf.concat([h] + ctxs, -1))
        return self.out(self.drop(feat, training=training)), state, w_d

    def call(self, inputs, training=False):
        hist, doc, dec_in = inputs                                 # teacher forcing
        enc, state = self.encode(hist, doc, training)
        logits = []
        for t in range(dec_in.shape[1]):
            lg, state, _ = self.step(dec_in[:, t], state, enc, training)
            logits.append(lg)
        return tf.stack(logits, axis=1)

In [54]:
## calculate loss
def masked_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    m = tf.cast(tf.not_equal(y_true, PAD), ce.dtype)
    return tf.reduce_sum(ce * m) / tf.reduce_sum(m)

In [55]:
## model training function
def train_model(use_doc, vocab, emb_matrix, train_ds, val_ds):
    model = DoGModel(len(vocab.itos), CFG["emb_dim"], CFG["units"], CFG["dropout"],
                     emb_matrix, use_doc=use_doc)
    model.compile(optimizer=tf.keras.optimizers.Adam(CFG["lr"], clipnorm=1.0), loss=masked_loss)
    model.fit(train_ds, validation_data=val_ds, epochs=CFG["epochs"], verbose=2,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    return model

In [56]:
def greedy_decode(model, hist, doc, max_len):
    enc, state = model.encode(hist, doc, False)
    B = hist.shape[0]
    block = np.zeros(len(model.out.get_weights()[1]), dtype="float32")
    block[[PAD, UNK, BOS]] = -1e9                                  # never generate these
    block = tf.constant(block)
    y, out = tf.fill([B], BOS), []
    for _ in range(max_len + 1):
        lg, state, _ = model.step(y, state, enc, False)
        y = tf.argmax(lg + block, axis=-1, output_type=tf.int32)
        out.append(y)
    return tf.stack(out, 1).numpy()


def generate(model, arrs, vocab, bs=128):
    hyps = []
    for i in range(0, len(arrs["hist"]), bs):
        ids = greedy_decode(model, tf.constant(arrs["hist"][i:i + bs]),
                            tf.constant(arrs["doc"][i:i + bs]), CFG["max_resp_len"])
        hyps += [vocab.dec(r) for r in ids]
    return hyps


#### Baseline

In [57]:
def retrieval_baseline(train_ex, test_ex):
    """Return the training reply whose history is most similar (TF-IDF cosine)."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import linear_kernel
    vec = TfidfVectorizer(token_pattern=r"\S+", sublinear_tf=True)
    X = vec.fit_transform([" ".join(e["hist"]) for e in train_ex])
    Q = vec.transform([" ".join(e["hist"]) for e in test_ex])
    best = linear_kernel(Q, X).argmax(1)
    return [" ".join(train_ex[j]["resp"]) for j in best]
# (the ungrounded neural baseline is simply DoGModel(use_doc=False) — see main())

#### Evaluation functions

In [58]:
def overlap_metrics(hyps, refs):
    import sacrebleu
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"])
    r = [sc.score(ref, h) for h, ref in zip(hyps, refs)]
    return dict(
        BLEU=sacrebleu.corpus_bleu(hyps, [refs], tokenize="none").score,
        R1=100 * np.mean([x["rouge1"].fmeasure for x in r]),
        RL=100 * np.mean([x["rougeL"].fmeasure for x in r]))

In [59]:
def distinct(hyps, n):
    g = [tuple(h.split()[i:i + n]) for h in hyps for i in range(len(h.split()) - n + 1)]
    return len(set(g)) / max(1, len(g))

In [60]:
def doc_overlap(hyps, docs_tok, common):
    """Share of a reply's content words (len>=4, not common) that occur in the document."""
    s = []
    for h, d in zip(hyps, docs_tok):
        c = [w for w in h.split() if len(w) >= 4 and w not in common]
        if c:
            ds = set(d)
            s.append(np.mean([w in ds for w in c]))
    return 100 * float(np.mean(s)) if s else float("nan")

In [61]:
def eng_frac(hi_en, en):
    """Fraction of Hinglish tokens that also appear in the English version of the same turn."""
    h = [w for w in tok(hi_en) if w.isalpha()]
    e = set(tok(en))
    return float(np.mean([w in e for w in h])) if h else float("nan")

In [62]:

def bucket(f):
    if np.isnan(f): return None
    return "mostly_english" if f >= 0.8 else ("code_mixed" if f > 0.2 else "mostly_hindi")

In [63]:
def evaluate(systems, test_ex, vocab):
    refs = [" ".join(e["resp"]) for e in test_ex]
    docs_tok = [e["doc"] for e in test_ex]
    g_idx = [i for i, e in enumerate(test_ex) if e["grounded"]]
    buckets = [bucket(eng_frac(" ".join(e["resp"]), e["en"])) for e in test_ex]
    rows = {}
    for name, hyps in systems.items():
        row = overlap_metrics(hyps, refs)
        row.update(D1=distinct(hyps, 1), D2=distinct(hyps, 2),
                   DocOverlap_grounded=doc_overlap([hyps[i] for i in g_idx],
                                                   [docs_tok[i] for i in g_idx], vocab.common))
        for b in ["mostly_english", "code_mixed", "mostly_hindi"]:
            idx = [i for i, x in enumerate(buckets) if x == b]
            if idx:
                row[f"RL[{b}]"] = overlap_metrics([hyps[i] for i in idx], [refs[i] for i in idx])["RL"]
        rows[name] = row
    refrow = dict(DocOverlap_grounded=doc_overlap([refs[i] for i in g_idx],
                                                  [docs_tok[i] for i in g_idx], vocab.common))
    rows["(human reference)"] = refrow
    return rows

#### doing on data

In [64]:
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); tf.random.set_seed(CFG["seed"])

In [65]:
docs = load_doc(CFG["doc_dir"])
convs = {s: load_conversations(s) for s in ["train", "validation", "test"]}
ex = {s: build_examples(c, docs, CFG["hist_turns"]) for s, c in convs.items()}

print({s: len(v) for s, v in ex.items()}, "examples")

{'train': 7855, 'validation': 915, 'test': 933} examples


In [66]:
utt_tokens = [tok(t["hi_en"]) for c in convs["train"].values() for t in c]
doc_tokens = [tok(section_text(d, s)) for d in docs.values() for s in range(4)]
vocab = Vocab(utt_tokens, doc_tokens, CFG["vocab_size"])
print("vocab size:", len(vocab.itos))

vocab size: 8339


In [67]:
emb = build_embedding_matrix(utt_tokens + doc_tokens, vocab, CFG["emb_dim"])
arr = {s: encode_examples(v, vocab, CFG) for s, v in ex.items()}
train_ds, val_ds = make_ds(arr["train"], CFG["batch"], True), make_ds(arr["validation"], CFG["batch"], False)

In [68]:
print("\n=== grounded model ===")
grounded = train_model(True, vocab, emb, train_ds, val_ds)
print("\n=== ungrounded baseline ===")
ungrounded = train_model(False, vocab, emb, train_ds, val_ds)


=== grounded model ===
Epoch 1/20
123/123 - 78s - 631ms/step - loss: 6.4968 - val_loss: 6.0899
Epoch 2/20
123/123 - 36s - 292ms/step - loss: 6.2550 - val_loss: 6.0979
Epoch 3/20
123/123 - 30s - 243ms/step - loss: 6.2499 - val_loss: 6.0989
Epoch 4/20
123/123 - 29s - 239ms/step - loss: 6.2499 - val_loss: 6.1072

=== ungrounded baseline ===
Epoch 1/20
123/123 - 48s - 392ms/step - loss: 6.5175 - val_loss: 6.0836
Epoch 2/20
123/123 - 18s - 144ms/step - loss: 6.2543 - val_loss: 6.0860
Epoch 3/20
123/123 - 14s - 117ms/step - loss: 6.2312 - val_loss: 6.0813
Epoch 4/20
123/123 - 14s - 116ms/step - loss: 6.2002 - val_loss: 6.0347
Epoch 5/20
123/123 - 14s - 116ms/step - loss: 6.1300 - val_loss: 5.9281
Epoch 6/20
123/123 - 14s - 115ms/step - loss: 6.0053 - val_loss: 5.7676
Epoch 7/20
123/123 - 14s - 115ms/step - loss: 5.7792 - val_loss: 5.4962
Epoch 8/20
123/123 - 14s - 115ms/step - loss: 5.4789 - val_loss: 5.2328
Epoch 9/20
123/123 - 21s - 167ms/step - loss: 5.1640 - val_loss: 5.0117
Epoch 10/20

In [69]:
systems = {
    "retrieval": retrieval_baseline(ex["train"], ex["test"]),
    "ungrounded_seq2seq": generate(ungrounded, arr["test"], vocab),
    "grounded_seq2seq": generate(grounded, arr["test"], vocab),
}
rows = evaluate(systems, ex["test"], vocab)
for name, row in rows.items():
    print(name, {k: round(v, 2) for k, v in row.items()})

retrieval {'BLEU': 0.31, 'R1': np.float64(5.72), 'RL': np.float64(5.49), 'D1': 0.15, 'D2': 0.43, 'DocOverlap_grounded': 4.86, 'RL[mostly_english]': np.float64(12.51), 'RL[code_mixed]': np.float64(5.61), 'RL[mostly_hindi]': np.float64(4.77)}
ungrounded_seq2seq {'BLEU': 0.46, 'R1': np.float64(8.24), 'RL': np.float64(7.65), 'D1': 0.01, 'D2': 0.03, 'DocOverlap_grounded': 3.83, 'RL[mostly_english]': np.float64(10.97), 'RL[code_mixed]': np.float64(6.92), 'RL[mostly_hindi]': np.float64(8.13)}
grounded_seq2seq {'BLEU': 0.0, 'R1': np.float64(0.0), 'RL': np.float64(0.0), 'D1': 0.0, 'D2': 0.0, 'DocOverlap_grounded': nan, 'RL[mostly_english]': np.float64(0.0), 'RL[code_mixed]': np.float64(0.0), 'RL[mostly_hindi]': np.float64(0.0)}
(human reference) {'DocOverlap_grounded': 11.15}


In [70]:
# qualitative samples: eyeball whether grounded replies use document facts
for i in random.sample(range(len(ex["test"])), 8):
    print("\nHIST :", " ".join(ex["test"][i]["hist"][-40:]))
    print("REF  :", " ".join(ex["test"][i]["resp"]))
    for n, h in systems.items():
        print(f"{n[:6].upper():5}:", h[i])

with open("results.json", "w") as f:
    json.dump(rows, f, indent=1)


HIST : family of toys since well , he is mr . potato head . hahaha . <sep> <s2> i always forget that woody had a girlfriend bopeep ! i heard the 4th movie they ’ re making is about <sep> <s2>
REF  : unhe khoj rahi hai
RETRIE: pizza planet ke bare mei kya kahana hai . abhi kya confusion hai ?
UNGROU: main bhi , mujhe lagta hai ki ye ek movie hai .
GROUND: 

HIST : <s1> i do , classic hollywood is a favorite <sep> <s1> i do , classic hollywood is a favorite <sep> <s2> wait i . t was actually 2005 ! i ’ m sorry <sep> <s2>
REF  : ruko . wo actually 2005 tha . i ’ m sorry
RETRIE: raat fury ?
UNGROU: main bhi , mujhe lagta hai ki ye movie bahut funny hai .
GROUND: 

HIST : <s1> same here <sep> <s2> yes <sep> <s2> it made me want to go back to see her <sep> <s1>
REF  : kam se kam maine do baar dekha , it was that good . mujhe female superheros bahut pasand hai
RETRIE: aapko pasand hein
UNGROU: haan , mujhe lagta hai ki ye movie bahut achha hai
GROUND: 

HIST : <s1> and the sequel as well <se